In [4]:
import os
import re
from typing import TypedDict
from typing import Literal
from dotenv import load_dotenv
from pydantic import BaseModel
from pydantic import Field
from pydantic import ValidationError

In [5]:
from langgraph.graph import StateGraph
from langgraph.graph import END
from langchain.tools import BaseTool
from langchain_google_genai import ChatGoogleGenerativeAI

In [6]:
# Load Gemini API Key
load_dotenv()

True

In [7]:
# ============================
# 1 Shared LangGraph State
# ============================
class ToolState(TypedDict):
    input: str
    selected_tool: Literal["math", "llm"]
    output: str

In [8]:
# ============================
# Define Pydantic Schema
# ============================
class MathSchema(BaseModel):
    a: float = Field(..., description="First number")
    b: float = Field(..., description="Second number")

In [ ]:
# ============================
# Custom Math Tool
# ============================
class AddNumbersTool(BaseTool):
    name: str = "add_numbers"
    description: str = "Adds two numbers using a validated schema"
    args_schema: type[BaseModel] = MathSchema

    def _run(self, a: float, b: float) -> str:
        return str(a + b)

    def _arun(self, *args, **kwargs):
        raise NotImplementedError()

add_tool = AddNumbersTool()

In [10]:
# ============================
# LLM (Gemini)
# ============================
llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro", temperature=0.7)

In [11]:
# ============================
# Tool Router Node
# ============================
def tool_router(state: ToolState) -> ToolState:
    query = state["input"]
    prompt = f"""
Decide whether this input needs math calculation or not.

If the input contains a numeric calculation, return 'math'.
Else return 'llm'.

Input: {query}
Reply with only one word: 'math' or 'llm'
"""
    decision = llm.invoke(prompt).content.strip().lower()
    print(f"Tool Selected: {decision}")
    state["selected_tool"] = "math" if "math" in decision else "llm"
    return state

In [12]:
# ============================
# Math Tool Node
# ============================
def math_node(state: ToolState) -> ToolState:
    query = state["input"]
    try:
        # Naive number extractor
        numbers = [float(n) for n in re.findall(r"\d+\.?\d*", query)]
        if len(numbers) < 2:
            raise ValueError("Need at least 2 numbers")

        result = add_tool.invoke({"a": numbers[0], "b": numbers[1]})
        state["output"] = f"Answer: {result}"
    except ValidationError as e:
        state["output"] = f"Schema Validation Error: {str(e)}"
    except Exception as e:
        state["output"] = f"Math Processing Error: {str(e)}"
    return state

In [13]:
# ============================
# Fallback LLM Node
# ============================
def llm_node(state: ToolState) -> ToolState:
    response = llm.invoke(state["input"]).content.strip()
    state["output"] = f"LLM Response: {response}"
    return state

In [15]:
# ============================
# LangGraph Setup
# ============================
builder = StateGraph(ToolState)
builder.set_entry_point("router")

builder.add_node("router", tool_router)
builder.add_node("math_tool", math_node)
builder.add_node("llm_tool", llm_node)

builder.add_conditional_edges(
    "router",
    lambda state: state["selected_tool"],
    {
        "math": "math_tool",
        "llm": "llm_tool"
    }
)

builder.add_edge("math_tool", END)
builder.add_edge("llm_tool", END)

In [16]:
graph = builder.compile()

In [17]:
# ============================
# Test Run
# ============================
examples = [
    {"input": "Add 34.5 and 21.7", "selected_tool": "", "output": ""},
    {"input": "What is the capital of Canada?", "selected_tool": "", "output": ""}
]

In [18]:
for example in examples:
    result = graph.invoke(example)
    print(f"\nFINAL OUTPUT:\n{result['output']}")

Tool Selected: math

FINAL OUTPUT:
Answer: 56.2
Tool Selected: llm

FINAL OUTPUT:
LLM Response: The capital of Canada is **Ottawa**.
